In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DateType, TimestampType, FloatType

In [0]:
catalog_name = 'ecommerce'

In [0]:
df = spark.table(f"{catalog_name}.silver.slv_order_items")

df.limit(10).display()

dt,order_ts,customer_id,order_id,item_seq,product_id,quantity,unit_price_currency,unit_price,discount_pct,tax_amount,channel,coupon_code,file_name,ingest_timestamp,processed_time
2025-08-01,2025-08-01T06:27:03.000Z,CUST000000043065,643621,1,2000000160061,1,INR,3344.0,6.0,379.0,Website,prime5,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-06-04T09:11:11.044Z,2026-06-04T16:04:37.820Z
2025-08-01,2025-08-01T21:07:10.000Z,CUST000000074149,643636,3,2000000225067,1,INR,6298.0,18.0,259.0,Website,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-06-04T09:11:11.044Z,2026-06-04T16:04:37.820Z
2025-08-01,2025-08-01T13:15:18.000Z,CUST000000146421,643647,1,2000000343594,1,USD,22.0,22.0,3.0,Mobile,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-06-04T09:11:11.044Z,2026-06-04T16:04:37.820Z
2025-08-01,2025-08-01T15:07:20.000Z,CUST000000030247,643653,1,2000000478913,1,INR,6244.0,3.0,1094.0,Mobile,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-06-04T09:11:11.044Z,2026-06-04T16:04:37.820Z
2025-08-01,2025-08-01T03:04:24.000Z,CUST000000093993,643658,1,2000000127804,1,USD,60.0,10.0,7.0,Mobile,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-06-04T09:11:11.044Z,2026-06-04T16:04:37.820Z
2025-08-01,2025-08-01T19:51:29.000Z,CUST000000154154,643669,2,2000000058986,1,INR,1040.0,25.0,95.0,Website,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-06-04T09:11:11.044Z,2026-06-04T16:04:37.820Z
2025-08-01,2025-08-01T02:12:31.000Z,CUST000000187150,643678,1,2000000250731,1,INR,2673.0,0.0,482.0,Website,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-06-04T09:11:11.044Z,2026-06-04T16:04:37.820Z
2025-08-01,2025-08-01T02:12:31.000Z,CUST000000187150,643678,2,2000000367095,1,INR,72286.0,6.0,8179.0,Website,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-06-04T09:11:11.044Z,2026-06-04T16:04:37.820Z
2025-08-01,2025-08-01T18:45:58.000Z,CUST000000126777,643688,4,2000000107721,3,AED,39.0,6.0,14.0,Website,prime5,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-06-04T09:11:11.044Z,2026-06-04T16:04:37.820Z
2025-08-01,2025-08-01T19:58:37.000Z,CUST000000216676,643696,2,2000000402604,1,INR,77408.0,14.0,8009.0,Mobile,new10,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-06-04T09:11:11.044Z,2026-06-04T16:04:37.820Z


In [0]:
# 1) Add gross amount
df = df.withColumn(
    "gross_amount",
    F.col("quantity") * F.col("unit_price")
    )

# 2) Add discount_amount (discount_pct is already numeric, e.g., 21 -> 21%)
df = df.withColumn(
    "discount_amount",
    F.ceil(F.col("gross_amount") * (F.col("discount_pct") / 100.0))
)

# 3) Add sale_amount = gross - discount
df = df.withColumn(
    "sale_amount",
    F.col("gross_amount") - F.col("discount_amount") + F.col("tax_amount")
)

# add date id
df = df.withColumn("date_id", F.date_format(F.col("dt"), "yyyyMMdd").cast(IntegerType()))  # Create date_key

# Coupon flag
#  coupon flag = 1 if coupon_code is not null else 0
df = df.withColumn(
    "coupon_flag",
    F.when(F.col("coupon_code").isNotNull(), F.lit(1))
     .otherwise(F.lit(0))
)

df.limit(5).display()    

dt,order_ts,customer_id,order_id,item_seq,product_id,quantity,unit_price_currency,unit_price,discount_pct,tax_amount,channel,coupon_code,file_name,ingest_timestamp,processed_time,gross_amount,discount_amount,sale_amount,date_id,coupon_flag
2025-08-01,2025-08-01T06:27:03.000Z,CUST000000043065,643621,1,2000000160061,1,INR,3344.0,6.0,379.0,Website,prime5,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-06-04T09:11:11.044Z,2026-06-04T16:04:37.820Z,3344.0,201,3522.0,20250801,1
2025-08-01,2025-08-01T21:07:10.000Z,CUST000000074149,643636,3,2000000225067,1,INR,6298.0,18.0,259.0,Website,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-06-04T09:11:11.044Z,2026-06-04T16:04:37.820Z,6298.0,1134,5423.0,20250801,0
2025-08-01,2025-08-01T13:15:18.000Z,CUST000000146421,643647,1,2000000343594,1,USD,22.0,22.0,3.0,Mobile,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-06-04T09:11:11.044Z,2026-06-04T16:04:37.820Z,22.0,5,20.0,20250801,0
2025-08-01,2025-08-01T15:07:20.000Z,CUST000000030247,643653,1,2000000478913,1,INR,6244.0,3.0,1094.0,Mobile,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-06-04T09:11:11.044Z,2026-06-04T16:04:37.820Z,6244.0,188,7150.0,20250801,0
2025-08-01,2025-08-01T03:04:24.000Z,CUST000000093993,643658,1,2000000127804,1,USD,60.0,10.0,7.0,Mobile,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-06-04T09:11:11.044Z,2026-06-04T16:04:37.820Z,60.0,6,61.0,20250801,0


currency conversion

In [0]:
# --- 1) Define your fixed FX rates (as of 2025-10-15, like your PBI note) ---
fx_rates = {
    "INR": 1.00,
    "AED": 24.18,
    "AUD": 57.55,
    "CAD": 62.93,
    "GBP": 117.98,
    "SGD": 68.18,
    "USD": 88.29,
}

rates = [(k, float(v)) for k, v in fx_rates.items()]
rates_df = spark.createDataFrame(rates, ["currency", "inr_rate"])
rates_df.show()

+--------+--------+
|currency|inr_rate|
+--------+--------+
|     INR|     1.0|
|     AED|   24.18|
|     AUD|   57.55|
|     CAD|   62.93|
|     GBP|  117.98|
|     SGD|   68.18|
|     USD|   88.29|
+--------+--------+

